## Structured output
Models can be requested to provide their response in a format matching a given schema. This is useful for
ensuring the output can be easily parsed and used in subsequent processing. LangChain supports multiple
schema types and methods for enforcing structured output.
## Pydantic
Pydantic models provide the richest feature set with field validation, descriptions, and nested structures.

In [28]:
import os
from langchain.chat_models import init_chat_model
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")
model = init_chat_model("groq:qwen/qwen3.6-27b")


In [ ]:
from pydantic import BaseModel,Field

class Movie(BaseModel):
    title:str = Field(description="The title of the movie")
    year:int = Field(description="This year the movie was released")
    director:str = Field(description="Director of the Movie")
    rating:float = Field(description="The movie's rating out of 10")

    


In [9]:
model_with_structured = model.with_structured_output(Movie)

In [10]:
model_with_structured.invoke("Provide details about the movie Avenger's End Game")

Movie(title='Avengers: Endgame', year=2019, director='Anthony and Joe Russo', rating=8)

## Message Output alongside parsed structure

In [ ]:
from pydantic import BaseModel,Field

class Movie(BaseModel):
    """ A movie with details. """
    title:str = Field(...,description="The title of the movie")
    year:int = Field(...,description="This year the movie was released")
    director:str = Field(...,description="Director of the Movie")
    rating:float = Field(...,description="The movie's rating out of 10")

model_with_structured = model.with_structured_output(Movie,include_raw=True)
response = model_with_structured.invoke("Provide details about the movie Avenger's End Game")
response

{'raw': AIMessage(content='', additional_kwargs={'reasoning_content': 'Here\'s a thinking process:\n\n1.  **Analyze User Input:**\n   - User wants details about the movie "Avenger\'s End Game"\n   - Note: The correct title is "Avengers: Endgame" (2019)\n   - I need to use the provided `Movie` function to provide details.\n\n2.  **Identify Required Parameters for `Movie` function:**\n   - `title` (string): "Avengers: Endgame"\n   - `year` (integer): 2019\n   - `director` (string): Anthony and Joe Russo (often listed as "Anthony Russo, Joe Russo" or "The Russo Brothers")\n   - `rating` (number): Need to provide a rating out of 10. Common ratings: IMDb ~8.4, Rotten Tomatoes ~94% (but that\'s not out of 10), Metacritic ~78/100. I\'ll use a widely accepted IMDb-style rating out of 10, like 8.4.\n\n3.  **Construct Function Call:**\n   - title: "Avengers: Endgame"\n   - year: 2019\n   - director: "Anthony and Joe Russo"\n   - rating: 8.4\n\n   Let\'s verify the details:\n   - Title: Avengers:

## Nested Structure

In [14]:
from pydantic import BaseModel,Field

class Actor(BaseModel):
    name:str
    role:str

class MovieDetails(BaseModel):
    title:str
    year:int
    cast:list[Actor]
    genres:list[str]
    budget:float | None = Field(None,description="Budget in millions USD")



In [15]:
model_with_structured = model.with_structured_output(MovieDetails)
response = model_with_structured.invoke("Provide details about the movie Avenger's End Game")
response

MovieDetails(title='Avengers: Endgame', year=2019, cast=[Actor(name='Robert Downey Jr.', role='Tony Stark / Iron Man'), Actor(name='Chris Evans', role='Steve Rogers / Captain America'), Actor(name='Scarlett Johansson', role='Natasha Romanoff / Black Widow'), Actor(name='Chris Hemsworth', role='Thor'), Actor(name='Mark Ruffalo', role='Bruce Banner / Hulk'), Actor(name='Jeremy Renner', role='Clint Barton / Hawkeye'), Actor(name='Josh Brolin', role='Thanos')], genres=['Action', 'Adventure', 'Sci-Fi'], budget=356.0)

# TypedDict

### TypedDict provides a simpler alternatinve using Python's built-in typing, ideal when you don't need runtime validation

In [19]:
from typing_extensions import  TypedDict,Annotated

class MovieDict(TypedDict):
    """A Movie with Details"""
    title: Annotated[str,...,"the title of the movie"]
    year:Annotated[int,...,"The year the movie was released"]
    director: Annotated[str,...,"The director of the movie"]
    rating:Annotated[float,...,"The movie's rating out of 10"]



In [17]:
model_with_typeDict = model.with_structured_output(MovieDict)
model_with_typeDict.invoke("Provide details about the movie Avenger's End Game")

{'director': 'Anthony Russo, Joe Russo',
 'rating': 8.4,
 'title': 'Avengers: Endgame',
 'year': 2019}

In [ ]:
from typing_extensions import  TypedDict,Annotated

class Actor(TypedDict):
    name:str
    role:str

class MovieDetails(TypedDict):
    title:str
    year:int
    cast:list[Actor]
    genres:list[str]
    budget:float | None = Field(None,description="Budget in millions USD")
    income:float

model_with_typeDict = model.with_structured_output(MovieDetails)
model_with_typeDict.invoke("Provide details about the movie Avenger's End Game")


{'budget': 356000000,
 'cast': [{'name': 'Robert Downey Jr.', 'role': 'Tony Stark / Iron Man'},
  {'name': 'Chris Evans', 'role': 'Steve Rogers / Captain America'},
  {'name': 'Mark Ruffalo', 'role': 'Bruce Banner / Hulk'},
  {'name': 'Chris Hemsworth', 'role': 'Thor'},
  {'name': 'Scarlett Johansson', 'role': 'Natasha Romanoff / Black Widow'},
  {'name': 'Jeremy Renner', 'role': 'Clint Barton / Hawkeye'}],
 'genres': ['Action', 'Adventure', 'Science Fiction'],
 'income': 2797501328,
 'title': 'Avengers: Endgame',
 'year': 2019}

## Data Classes

#### A data class is a class typically containing mainly data, although there are not really any restrictions. You create it using the @dataclass decorator

In [ ]:
import os
from langchain.chat_models import init_chat_model
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")
model = init_chat_model("groq:qwen/qwen3.6-27b")

In [ ]:
from pydantic import BaseModel, Field
from langchain_groq import ChatGroq


class ContactInfo(BaseModel):
    """Contact information for a person"""

    name: str = Field(description="The name of the person")
    email: str = Field(description="The email address of the person")
    phone: int = Field(description="The phone number of the person")


llm = ChatGroq(
    model="qwen/qwen3.6-27b",
)

structured_llm = llm.with_structured_output(ContactInfo)

result = structured_llm.invoke(
    "Extract contact info from: john doe, abc@xyz.com, '+91-8899554785'"
)

print(result)
print(type(result))

name='john doe' email='abc@xyz.com' phone=55599554785
<class '__main__.ContactInfo'>


In [49]:
from typing_extensions import  TypedDict,Annotated
from langchain_groq import ChatGroq


class ContactInfo(TypedDict):
    """Contact information for a person"""
    name: str 
    email: str 
    phone: str


llm = ChatGroq(
    model="qwen/qwen3.6-27b",
)

structured_llm = llm.with_structured_output(ContactInfo)

result = structured_llm.invoke(
    "Extract contact info from: john doe, abc@xyz.com, '+91-8899554785'"
)

print(result)
print(type(result))

{'email': 'abc@xyz.com', 'name': 'john doe', 'phone': '+91-8899554785'}
<class 'dict'>


In [65]:
##DataClasses

from dataclasses import dataclass
from langchain_groq import ChatGroq
from langchain.agents import create_agent

@dataclass
class ContactInfo:
        """Contact information for a person"""
        name: str 
        email: str 
        phone: str

llm = ChatGroq(
    model="qwen/qwen3.6-27b"
)


agent = create_agent(
    model=llm,
    response_format=ContactInfo
)

result = agent.invoke({
    "messages": [
        (
            "user",
            "Extract contact info from: john doe, abc@xyz.com, '+91-8899554785'"
        )
    ]
})

print(result["structured_response"])
print(type(result["structured_response"]))

ContactInfo(name='john doe', email='abc@xyz.com', phone='+91-8899554785')
<class '__main__.ContactInfo'>
